In [0]:
%sql
SELECT *
FROM healthcare_dev.bronze.hospitals_raw;

In [0]:
# ============================================================
# SILVER LAYER — DIM_HOSPITAL
# Source: healthcare_dev.bronze.hospitals_raw
# Target: healthcare_dev.silver.dim_hospital
# ============================================================

from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

# 1. Configure Bronze source, Silver target, and checkpoint

bronze_table = "healthcare_dev.bronze.hospitals_raw"
silver_table = "healthcare_dev.silver.dim_hospital"

checkpoint_path = (
    "abfss://healthcare@ecommerceunifiedproject.dfs.core.windows.net/"
    "silver/dim_hospital/checkpoint/"
)

# 2. Read Bronze using Structured Streaming

df_hospital_bronze = spark.readStream.table(bronze_table)

# 3. Deduplicate hospital records and add load timestamp

df_hospital_clean = (
    df_hospital_bronze
    .dropDuplicates(["hospital_id"])
    .withColumn("load_timestamp", current_timestamp())
)

# 4. Define Delta MERGE operation

def merge_dim_hospital(batch_df, batch_id):

    if not spark.catalog.tableExists(silver_table):
        (
            batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        return

    dim_hospital = DeltaTable.forName(spark, silver_table)

    (
        dim_hospital.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.hospital_id = s.hospital_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# 5. Execute streaming ingestion

query = (
    df_hospital_clean.writeStream
    .foreachBatch(merge_dim_hospital)
    .outputMode("update")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

query.awaitTermination()

print("Silver dim_hospital ingestion completed.")

# 6. Verify Silver table

display(
    spark.table("healthcare_dev.silver.dim_hospital")
)